In [ ]:
import uproot
import os
import dask_awkward as dak
import awkward as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import mplhep as hep
from dask.diagnostics import ProgressBar
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
import joblib

In [ ]:
plt.style.use(hep.style.CMS)

In [ ]:
tree_path = "Events"
data_files = {f: tree_path for f in [
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022C.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022D.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022E.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022F.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_dados_Run2022G.root"
]}

mc_files = {f: tree_path for f in [
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_mc_signal_2022.root",
    "/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_mc_signal_2022EE.root"
]}

In [ ]:
def get_b0_truth_mask(df):
    # --- 1. Constantes PDG ---
    PDG_B0        = 511
    PDG_KSTAR     = 313
    PDG_KAON_plus = 321
    PDG_PION_neg  = -211
    PDG_MUON_plus = -13
    PDG_MUON_neg  = 13
    PDG_PHOTON    = 22

    mu1_idx  = df["BPH_1Muon_genPartIdx"]
    mu2_idx  = df["BPH_2Muon_genPartIdx"]
    trk1_idx = df["Trk1_genPartIdx"]
    trk2_idx = df["Trk2_genPartIdx"]

    # Verificação de Identidade (PDG ID)
    mu1_pdg  = df["BPHGenPart_pdgId"][dak.mask(mu1_idx, mu1_idx >= 0)]
    mu2_pdg  = df["BPHGenPart_pdgId"][dak.mask(mu2_idx, mu2_idx >= 0)]
    trk1_pdg = df["BPHGenPart_pdgId"][dak.mask(trk1_idx, trk1_idx >= 0)]
    trk2_pdg = df["BPHGenPart_pdgId"][dak.mask(trk2_idx, trk2_idx >= 0)]

    flavor_match = (trk1_pdg == PDG_KAON_plus) & (trk2_pdg == PDG_PION_neg) & \
                   (mu1_pdg == PDG_MUON_plus) & (mu2_pdg == PDG_MUON_neg)

    # Linhagem do Hadron (K, Pi -> K* -> B0)
    mom_trk1_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(trk1_idx, trk1_idx >= 0)]
    mom_trk2_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(trk2_idx, trk2_idx >= 0)]
    
    mom_trk1_pdg = df["BPHGenPart_pdgId"][dak.mask(mom_trk1_idx, mom_trk1_idx >= 0)]
    mom_trk2_pdg = df["BPHGenPart_pdgId"][dak.mask(mom_trk2_idx, mom_trk2_idx >= 0)]
    
    # K e Pi devem vir do mesmo objeto K*0 
    match_kstar = (mom_trk1_idx == mom_trk2_idx) & (mom_trk1_idx >= 0) & \
                  (mom_trk1_pdg == PDG_KSTAR) & (mom_trk2_pdg == PDG_KSTAR)
    
    # O K*0 deve vir de um B0 (Verificando para ambos os traços)
    gmom_trk1_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mom_trk1_idx, mom_trk1_idx >= 0)]
    gmom_trk2_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mom_trk2_idx, mom_trk2_idx >= 0)]
    
    gmom_trk1_pdg = df["BPHGenPart_pdgId"][dak.mask(gmom_trk1_idx, gmom_trk1_idx >= 0)]
    gmom_trk2_pdg = df["BPHGenPart_pdgId"][dak.mask(gmom_trk2_idx, gmom_trk2_idx >= 0)]
    
    match_kstar_to_b0 = (gmom_trk1_idx == gmom_trk2_idx) & (gmom_trk1_idx >= 0) & \
                        (gmom_trk1_pdg == PDG_B0) & (gmom_trk2_pdg == PDG_B0)

    mom_mu1_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mu1_idx, mu1_idx >= 0)]
    mom_mu2_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mu2_idx, mu2_idx >= 0)]
    mom_mu1_pdg = df["BPHGenPart_pdgId"][dak.mask(mom_mu1_idx, mom_mu1_idx >= 0)]
    mom_mu2_pdg = df["BPHGenPart_pdgId"][dak.mask(mom_mu2_idx, mom_mu2_idx >= 0)]
    
    gmom_mu1_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mom_mu1_idx, mom_mu1_idx >= 0)]
    gmom_mu2_idx = df["BPHGenPart_genPartIdxMother"][dak.mask(mom_mu2_idx, mom_mu2_idx >= 0)]

    # Muon 1: Mãe é o B0 do hadron OU (Mãe é fóton e Avô é o B0 do hadron)
    mu1_from_same_b0 = (mom_mu1_idx == gmom_trk1_idx) | \
                       ((mom_mu1_pdg == PDG_PHOTON) & (gmom_mu1_idx == gmom_trk1_idx))

    mu2_from_same_b0 = (mom_mu2_idx == gmom_trk1_idx) | \
                       ((mom_mu2_pdg == PDG_PHOTON) & (gmom_mu2_idx == gmom_trk1_idx))

    # --- Máscara Final ---
    full_mask = flavor_match & match_kstar & match_kstar_to_b0 & \
                mu1_from_same_b0 & mu2_from_same_b0
    
    return ak.fill_none(full_mask, False)

In [ ]:
def add_derived_columns(df):
    df["BToTrkTrkMuMu_l_xy_sig"] = df['BToTrkTrkMuMu_l_xy'] / df['BToTrkTrkMuMu_l_xy_unc']
    df["BToTrkTrkMuMu_dca_sig"]  = df['BToTrkTrkMuMu_dca'] / df['BToTrkTrkMuMu_dcaErr']
    df["BToTrkTrkMuMu_trk1_dca_sig"] = df['BToTrkTrkMuMu_trk1_dca'] / df['BToTrkTrkMuMu_trk1_dcaErr']
    df["BToTrkTrkMuMu_trk2_dca_sig"] = df['BToTrkTrkMuMu_trk2_dca'] / df['BToTrkTrkMuMu_trk2_dcaErr']
    return df

In [ ]:
def apply_selection(df, mode="mc"):

    kstar_pdg = 0.892
    mask_kpi_window = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_Kpi'] - kstar_pdg) <= 0.150
    mask_pik_window = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_piK'] - kstar_pdg) <= 0.150
    is_kpi_closer = abs(df['BToTrkTrkMuMu_fit_ditrack_mass_Kpi'] - kstar_pdg) < \
                    abs(df['BToTrkTrkMuMu_fit_ditrack_mass_piK'] - kstar_pdg)
    
    mask = (
        # Filtro de massa de dimuons (regiões de sinal, excluindo J/psi e Psi2S)
        (((df['MuMu_mass'] > 1.0) & (df['MuMu_mass'] < 2.7)) | 
         ((df['MuMu_mass'] > 4.0) & (df['MuMu_mass'] < 6.0))) & 
        
        # Pelo menos uma das combinações deve estar na janela de 3 sigma
        (mask_kpi_window | mask_pik_window) & 
        
        # Aplicamos a lógica de decisão de sabor (escolhemos a hipótese Kpi se ela for a melhor)
        (is_kpi_closer) &
        
        # Trigger e cortes cinemáticos dos múons
        (df['HLT_DoubleMu4_3_LowMass'] == 1) & 
        (df['BPH_1Muon_pt'] > 4.0) & (df['BPH_2Muon_pt'] > 4.0) & 
        (abs(df['BPH_1Muon_eta']) < 2.4) & (abs(df['BPH_2Muon_eta']) < 2.4)
    )
    if mode == "mc":
        mask = mask & (df['BToTrkTrkMuMu_fit_mass_Kpi'] >= 5.133542769) & (df['BToTrkTrkMuMu_fit_mass_Kpi'] <= 5.416657231)
        mask = mask & get_b0_truth_mask(df) # No MC junta a mask de thruth Matching !!!
    else:
        mask = mask & (
            ((df['BToTrkTrkMuMu_fit_mass_Kpi'] > 5.0) & (df['BToTrkTrkMuMu_fit_mass_Kpi'] < 5.133542769)) | 
            ((df['BToTrkTrkMuMu_fit_mass_Kpi'] > 5.416657231) & (df['BToTrkTrkMuMu_fit_mass_Kpi'] < 5.6))
        )
    
    cols_to_keep = [
        'BToTrkTrkMuMu_l_xy_sig', 'BToTrkTrkMuMu_dca_sig', 
        'BToTrkTrkMuMu_trk1_dca_sig', 'BToTrkTrkMuMu_trk2_dca_sig', 
        'BToTrkTrkMuMu_fit_ditrack_mass_Kpi', 'BToTrkTrkMuMu_fit_pt', 
        'BToTrkTrkMuMu_fit_cos2D', 'BToTrkTrkMuMu_svprob', 'event'
    ]
    return df[cols_to_keep][mask]

In [ ]:
def build_dataframe(file_dict, label="Dataset", mode="mc"):
    print(f"\nProcessando {label}...")
    
    df = uproot.dask(file_dict)
    df = add_derived_columns(df)
    df = apply_selection(df, mode=mode)
    
    with ProgressBar():
        awkward_array = df.compute()
        df_pandas = ak.to_dataframe(awkward_array).reset_index(drop=True)
    
    print(f"{label} finalizado. Linhas: {len(df_pandas)}")
    return df_pandas

In [ ]:
df_data = build_dataframe(data_files, "Data (Sidebands)", mode="data")
df_mc   = build_dataframe(mc_files, "MC Signal (Truth Matched)", mode="mc")

In [ ]:
df_mc['target'] = 1
df_data['target'] = 0

In [ ]:
df_total = pd.concat([df_mc, df_data], ignore_index=True)

In [ ]:
features = [
    'BToTrkTrkMuMu_l_xy_sig', 'BToTrkTrkMuMu_dca_sig', 
    'BToTrkTrkMuMu_trk1_dca_sig', 'BToTrkTrkMuMu_trk2_dca_sig', "BToTrkTrkMuMu_fit_ditrack_mass_Kpi",
    'BToTrkTrkMuMu_fit_pt', 'BToTrkTrkMuMu_fit_cos2D', 'BToTrkTrkMuMu_svprob'
]

In [ ]:
df_total.replace([np.inf, -np.inf], np.nan, inplace=True)
df_total.dropna(subset=features, inplace=True)

In [ ]:
df_total['fold_id'] = df_total['event'] % 11

In [ ]:
def get_max_punzi_significance(df, a):
    """
    Calcula a Significância de Punzi máxima varrendo os possíveis cortes de BDT.
    (Conforme CMS AN-18-138 Sec 4.1.3)
    """
    total_S = len(df[df['target'] == 1])
    if total_S == 0: return 0.0

    thresholds = np.linspace(0.1, 1, 100)
    best_punzi = 0.0
    
    for th in thresholds:
        pass_mask = df['bdt_score'] > th
        S_pass = len(df[(df['target'] == 1) & pass_mask])
        B_pass = len(df[(df['target'] == 0) & pass_mask])
        
        eps_S = S_pass / total_S
        punzi = eps_S / ((a / 2.0) + np.sqrt(B_pass))
        
        if punzi > best_punzi:
            best_punzi = punzi
            
    return best_punzi

In [ ]:
def objective(trial, df_total, features):
    """
    A Função Objetivo para o Optuna. 
    Para cada tentativa de hiperparâmetros, ela treina 11 BDTs cegas.
    """
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',      
        'device': 'cuda',
        'scale_pos_weight': 2.6,
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 10.0),
        'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
        'random_state': 42,
        'n_jobs': -1
    }

    df_temp = df_total[['target', 'fold_id']].copy()
    df_temp['bdt_score'] = -1.0
    
    # K-Fold de 11 subamostras conforme AN
    for i in range(11):
        mask_test = df_total['fold_id'] == i
        df_analysis = df_total[mask_test]       # 1 fatia blindada
        df_train_val = df_total[~mask_test]     # 10 fatias para treino
        
        # Divisão interna (70/30) das 10 fatias para early stopping
        X_tv = df_train_val[features]
        y_tv = df_train_val['target']
        X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.3, random_state=42, stratify=y_tv)

        clf = xgb.XGBClassifier(**params, early_stopping_rounds=30)
        
        clf.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False 
        )
        
        # Aplica a pontuação SOMENTE na fatia que não participou do treino
        df_temp.loc[mask_test, 'bdt_score'] = clf.predict_proba(df_analysis[features])[:, 1]
    
    return get_max_punzi_significance(df_temp, a=2.0)


In [ ]:
sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(lambda trial: objective(trial, df_total, features), n_trials=150, show_progress_bar=True)

In [ ]:
print("\n=======================================================")
print(f"Melhor Significância de Punzi (WP Opt): {study.best_value:.4f}")
print("Hiperparâmetros Ótimos Selecionados:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
print("=======================================================\n")

In [ ]:
best_params = study.best_params
best_params['objective'] = 'binary:logistic'
best_params['eval_metric'] = ['logloss', 'auc']
best_params['random_state'] = 42
best_params['n_jobs'] = -1

In [ ]:
os.makedirs("bdt_models_final", exist_ok=True)

In [ ]:
df_total['final_bdt_score'] = -1.0

In [ ]:
all_histories = []

print("\nIniciando o Treinamento Final dos 11 Folds...")
for i in range(11):
    mask_test = df_total['fold_id'] == i
    df_analysis = df_total[mask_test]       
    df_train_val = df_total[~mask_test]     
    
    X_tv = df_train_val[features]
    y_tv = df_train_val['target']
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.3, random_state=42, stratify=y_tv)
    
    clf = xgb.XGBClassifier(**best_params, early_stopping_rounds=30)
    
    clf.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False
    )
    
    all_histories.append(clf.evals_result())    
    df_total.loc[mask_test, 'final_bdt_score'] = clf.predict_proba(df_analysis[features])[:, 1]
    
    model_filename = f"bdt_models_final/xgboost_fold_{i}.joblib"
    joblib.dump(clf, model_filename)
    print(f"  -> Fold {i} treinado (Early stop na árvore {clf.best_iteration}) - salvo em {model_filename}")

In [ ]:
# ==============================================================================
# 1. PREPARANDO A CURVA DE APRENDIZAGEM MÉDIA (AUC vs Trees)
# ==============================================================================
max_epochs = max([len(h['validation_0']['auc']) for h in all_histories])

train_auc_matrix = np.zeros((11, max_epochs))
val_auc_matrix = np.zeros((11, max_epochs))

for i, h in enumerate(all_histories):
    t_auc = h['validation_0']['auc']
    v_auc = h['validation_1']['auc']
    
    train_auc_matrix[i, :len(t_auc)] = t_auc
    val_auc_matrix[i, :len(v_auc)] = v_auc
    
    if len(t_auc) < max_epochs:
        train_auc_matrix[i, len(t_auc):] = t_auc[-1]
        val_auc_matrix[i, len(v_auc):] = v_auc[-1]

mean_train_auc_history = np.mean(train_auc_matrix, axis=0)
mean_val_auc_history = np.mean(val_auc_matrix, axis=0)

# ==============================================================================
# 2. PREPARANDO A CURVA ROC MÉDIA (Train vs Test)
# ==============================================================================
tprs_train = []
tprs_test = []
mean_fpr = np.linspace(0, 1, 100) 

for i in range(11):
    mask_test = df_total['fold_id'] == i
    df_test = df_total[mask_test]
    df_train = df_total[~mask_test]
    
    model = joblib.load(f"bdt_models_final/xgboost_fold_{i}.joblib")
    
    y_train_pred = model.predict_proba(df_train[features])[:, 1]
    y_test_pred = model.predict_proba(df_test[features])[:, 1]
    y_train_true = df_train['target']
    y_test_true = df_test['target']
    
    fpr_train, tpr_train, _ = roc_curve(y_train_true, y_train_pred)
    interp_tpr_train = np.interp(mean_fpr, fpr_train, tpr_train)
    interp_tpr_train[0] = 0.0
    tprs_train.append(interp_tpr_train)
    
    fpr_test, tpr_test, _ = roc_curve(y_test_true, y_test_pred)
    interp_tpr_test = np.interp(mean_fpr, fpr_test, tpr_test)
    interp_tpr_test[0] = 0.0
    tprs_test.append(interp_tpr_test)

mean_tpr_train = np.mean(tprs_train, axis=0)
mean_tpr_train[-1] = 1.0
mean_auc_train = auc(mean_fpr, mean_tpr_train)

mean_tpr_test = np.mean(tprs_test, axis=0)
mean_tpr_test[-1] = 1.0
mean_auc_test = auc(mean_fpr, mean_tpr_test)

# ==============================================================================
# 3. PLOTAGEM DOS GRÁFICOS (ESTILO CMS AN)
# ==============================================================================
fig, ax = plt.subplots(1, 2, figsize=(16, 7))

# --- GRÁFICO 1: Curva de Aprendizagem Média (AGORA EM PORCENTAGEM) ---

# Converte o eixo x para percentagem (de 0 a 100%)
epochs_percent = np.linspace(0, 100, max_epochs)

ax[0].plot(epochs_percent, mean_train_auc_history, color='orange', lw=2.5, label='Training sample')
ax[0].plot(epochs_percent, mean_val_auc_history, color='violet', lw=2.5, label='Test sample')

# Adiciona o símbolo de '%' ao eixo X
ax[0].xaxis.set_major_formatter(PercentFormatter(100))

ax[0].set_xlabel('Percentage of trees (%)', fontsize=14)
ax[0].set_ylabel('Average AUC', fontsize=14)
ax[0].set_title('Average AUC over the 11 subsamples', fontsize=16)
ax[0].legend(loc='lower right', fontsize=12)
ax[0].grid(True, linestyle=':', alpha=0.6)

# --- GRÁFICO 2: Curva ROC Média ---
ax[1].plot([0, 1], [0, 1], linestyle='--', lw=2, color='gray', alpha=0.8)

ax[1].plot(mean_fpr, mean_tpr_train, color='orange', lw=2.5, 
           label=f'Training sample (AUC = {mean_auc_train:.3f})')

ax[1].plot(mean_fpr, mean_tpr_test, color='violet', lw=2.5, 
           label=f'Test sample (AUC = {mean_auc_test:.3f})')

ax[1].set_xlabel('Background efficiency (FPR)', fontsize=14)
ax[1].set_ylabel('Signal efficiency (TPR)', fontsize=14)
ax[1].set_title('ROC curve (averaged over 11 subsamples)', fontsize=16)
ax[1].legend(loc='lower right', fontsize=12)
ax[1].grid(True, linestyle=':', alpha=0.6)
#ax[1].set_xscale('log')
ax[1].set_xlim([0, 1.02])
ax[1].set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig("bdt_averaged_roc_and_auc_percent.png", dpi=300)
plt.show()

In [ ]:
# ==============================================================================
# 1. PREPARANDO A CURVA DE APRENDIZAGEM MÉDIA (AUC vs Trees)
# ==============================================================================
max_epochs = max([len(h['validation_0']['auc']) for h in all_histories])

train_auc_matrix = np.zeros((11, max_epochs))
val_auc_matrix = np.zeros((11, max_epochs))

for i, h in enumerate(all_histories):
    t_auc = h['validation_0']['auc']
    v_auc = h['validation_1']['auc']
    
    train_auc_matrix[i, :len(t_auc)] = t_auc
    val_auc_matrix[i, :len(v_auc)] = v_auc
    
    if len(t_auc) < max_epochs:
        train_auc_matrix[i, len(t_auc):] = t_auc[-1]
        val_auc_matrix[i, len(v_auc):] = v_auc[-1]

mean_train_auc_history = np.mean(train_auc_matrix, axis=0)
mean_val_auc_history = np.mean(val_auc_matrix, axis=0)

# ==============================================================================
# 2. PREPARANDO A CURVA ROC MÉDIA (Train vs Test)
# ==============================================================================
tprs_train = []
tprs_test = []

# MUDANÇA IMPORTANTE: Criamos 500 pontos em escala logarítmica de 10^-3 (0.001) a 10^0 (1)
# E adicionamos o 0.0 na primeira posição apenas para o cálculo correto da AUC
mean_fpr = np.insert(np.logspace(-3, 0, 500), 0, 0.0)

for i in range(11):
    mask_test = df_total['fold_id'] == i
    df_test = df_total[mask_test]
    df_train = df_total[~mask_test]
    
    model = joblib.load(f"bdt_models_final/xgboost_fold_{i}.joblib")
    
    y_train_pred = model.predict_proba(df_train[features])[:, 1]
    y_test_pred = model.predict_proba(df_test[features])[:, 1]
    y_train_true = df_train['target']
    y_test_true = df_test['target']
    
    fpr_train, tpr_train, _ = roc_curve(y_train_true, y_train_pred)
    interp_tpr_train = np.interp(mean_fpr, fpr_train, tpr_train)
    interp_tpr_train[0] = 0.0
    tprs_train.append(interp_tpr_train)
    
    fpr_test, tpr_test, _ = roc_curve(y_test_true, y_test_pred)
    interp_tpr_test = np.interp(mean_fpr, fpr_test, tpr_test)
    interp_tpr_test[0] = 0.0
    tprs_test.append(interp_tpr_test)

mean_tpr_train = np.mean(tprs_train, axis=0)
mean_tpr_train[-1] = 1.0
mean_auc_train = auc(mean_fpr, mean_tpr_train)

mean_tpr_test = np.mean(tprs_test, axis=0)
mean_tpr_test[-1] = 1.0
mean_auc_test = auc(mean_fpr, mean_tpr_test)

# ==============================================================================
# 3. PLOTAGEM DOS GRÁFICOS (ESTILO CMS AN)
# ==============================================================================
fig, ax = plt.subplots(1, 2, figsize=(16, 7))

# --- GRÁFICO 1: Curva de Aprendizagem Média ---
epochs_percent = np.linspace(0, 100, max_epochs)

ax[0].plot(epochs_percent, mean_train_auc_history, color='orange', lw=2.5, label='Training sample')
ax[0].plot(epochs_percent, mean_val_auc_history, color='violet', lw=2.5, label='Test sample')

ax[0].xaxis.set_major_formatter(PercentFormatter(100))

ax[0].set_xlabel('Percentage of trees (%)', fontsize=14)
ax[0].set_ylabel('Average AUC', fontsize=14)
ax[0].set_title('Average AUC over the 11 subsamples', fontsize=16)
ax[0].legend(loc='lower right', fontsize=12)
ax[0].grid(True, linestyle=':', alpha=0.6)

# --- GRÁFICO 2: Curva ROC Média (Escala Logarítmica) ---

# A linha de referência foi corrigida para não começar em 0 (matematicamente inválido em log)
ax[1].plot([0.001, 1], [0.001, 1], linestyle='--', lw=2, color='gray', alpha=0.8)

ax[1].plot(mean_fpr, mean_tpr_train, color='orange', lw=2.5, 
           label=f'Training sample (AUC = {mean_auc_train:.3f})')

ax[1].plot(mean_fpr, mean_tpr_test, color='violet', lw=2.5, 
           label=f'Test sample (AUC = {mean_auc_test:.3f})')

ax[1].set_xlabel('Background efficiency (FPR)', fontsize=14)
ax[1].set_ylabel('Signal efficiency (TPR)', fontsize=14)
ax[1].set_title('ROC curve (averaged over 11 subsamples)', fontsize=16)
ax[1].legend(loc='lower right', fontsize=12)

# Adicionando um grid com subdivisões para a escala logarítmica ficar visível
ax[1].grid(True, which="both", linestyle=':', alpha=0.6)

# --- APLICAÇÃO DA ESCALA LOG E DOS LIMITES ---
ax[1].set_xscale('log')
ax[1].set_xlim([0.001, 1.0])
ax[1].set_ylim([0.0, 1.02])

plt.tight_layout()
plt.savefig("bdt_averaged_roc_log_and_auc_percent.png", dpi=300)
plt.show()

In [ ]:
# ==============================================================================
# PREPARANDO A CURVA DE LOG LOSS MÉDIA
# ==============================================================================
# Encontra o limite máximo de árvores entre os 11 modelos
max_epochs = max([len(h['validation_0']['logloss']) for h in all_histories])

train_loss_matrix = np.zeros((11, max_epochs))
val_loss_matrix = np.zeros((11, max_epochs))

for i, h in enumerate(all_histories):
    t_loss = h['validation_0']['logloss']
    v_loss = h['validation_1']['logloss']
    
    # Preenchemos com os valores reais
    train_loss_matrix[i, :len(t_loss)] = t_loss
    val_loss_matrix[i, :len(v_loss)] = v_loss
    
    # Se parou mais cedo, assumimos que a loss estabilizou no último valor
    if len(t_loss) < max_epochs:
        train_loss_matrix[i, len(t_loss):] = t_loss[-1]
        val_loss_matrix[i, len(v_loss):] = v_loss[-1]

# Calcula as médias (Average) ao longo dos 11 folds
mean_train_loss_history = np.mean(train_loss_matrix, axis=0)
mean_val_loss_history = np.mean(val_loss_matrix, axis=0)

# ==============================================================================
# PLOTAGEM DO GRÁFICO (EIXO X EM PERCENTAGEM)
# ==============================================================================
plt.figure(figsize=(8, 6))

# Eixo x uniformemente distribuído de 0 a 100%
epochs_percent = np.linspace(0, 100, max_epochs)

# Mantive as cores laranja e violeta para manter a consistência visual com o plot da AUC
plt.plot(epochs_percent, mean_train_loss_history, color='orange', lw=1.5, label='Training sample')
plt.plot(epochs_percent, mean_val_loss_history, color='violet', lw=1.5, label='Test sample')

# Formata o eixo X para mostrar o símbolo '%'
plt.gca().xaxis.set_major_formatter(PercentFormatter(100))

plt.xlabel('Percentage of trees (%)', fontsize=14)
plt.ylabel('Average Log Loss', fontsize=14)
plt.title('Average Log Loss over the 11 subsamples', fontsize=16)

# Na Loss, as curvas descem, por isso a legenda fica melhor no canto superior direito
plt.legend(loc='upper right', fontsize=12) 
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.savefig("bdt_averaged_logloss_percent.png", dpi=300)
plt.show()

In [ ]:
# ==============================================================================
# 1. EXTRAÇÃO E NORMALIZAÇÃO DOS F-SCORES (WEIGHTS) DOS 11 MODELOS
# ==============================================================================
# Dicionário para guardar as importâncias normalizadas de cada variável em todos os folds
normalized_scores_all_folds = {feat: [] for feat in features}

for i in range(11):
    # Carrega o modelo de cada fold salvo anteriormente
    model = joblib.load(f"bdt_models_final/xgboost_fold_{i}.joblib")
    
    # IMPORTANTE: extrai especificamente o 'weight' (F-score = número de aparições nas árvores)
    # Não usamos feature_importances_ padrão porque ele retornaria o 'gain'
    booster = model.get_booster()
    f_scores = booster.get_score(importance_type='weight')
    
    # Como algumas variáveis podem não ser usadas em nenhuma árvore (raro, mas possível),
    # usamos .get(feat, 0.0) para garantir que não dê erro de chave (KeyError).
    fold_scores = np.array([f_scores.get(feat, 0.0) for feat in features])
    
    # "normalized to the highest F-score of the specific sample"
    max_score = np.max(fold_scores)
    
    if max_score > 0:
        fold_scores_norm = fold_scores / max_score
    else:
        fold_scores_norm = fold_scores
        
    # Guarda o valor normalizado para calcular a média depois
    for feat, score in zip(features, fold_scores_norm):
        normalized_scores_all_folds[feat].append(score)

# ==============================================================================
# 2. CÁLCULO DA MÉDIA (AVERAGE) 
# ==============================================================================
# Calcula a média (over the 11 sub-samples)
average_f_scores = {feat: np.mean(scores) for feat, scores in normalized_scores_all_folds.items()}

# Ordena as variáveis da menor para a maior importância (para o plot de barras horizontais ficar com a maior no topo)
sorted_features = sorted(average_f_scores.items(), key=lambda x: x[1], reverse=False)

feat_names = [x[0] for x in sorted_features]
feat_scores = [x[1] for x in sorted_features]

# ==============================================================================
# 3. PLOTAGEM DO GRÁFICO (Feature Importance F-Score)
# ==============================================================================
plt.figure(figsize=(12, 8))

# Gráfico de barras horizontais
bars = plt.barh(feat_names, feat_scores, color='cornflowerblue', edgecolor='black', height=0.7)

# Adiciona o valor numérico exato no final de cada barra para facilitar a leitura
for bar in bars:
    plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
             f'{bar.get_width():.3f}', 
             va='center', ha='left', fontsize=11)

plt.xlabel('Average normalized F-score', fontsize=14)
plt.ylabel('BDT Input Variables', fontsize=14)
plt.title('Average F-score of BDT input variables (11 subsamples)', fontsize=16)

# Limite do eixo X vai até 1.1 para dar espaço para o texto dos números
plt.xlim(0, 1.1)
plt.grid(axis='x', linestyle=':', alpha=0.7)

plt.tight_layout()
plt.savefig("bdt_average_f_score_importance.png", dpi=300)
plt.show()

In [ ]:
fold_idx = 0

mask_test = df_total['fold_id'] == fold_idx
df_test = df_total[mask_test]
df_train = df_total[~mask_test]

model = joblib.load(f"bdt_models_final/xgboost_fold_{fold_idx}.joblib")

y_probs_train = model.predict_proba(df_train[features])[:, 1]
y_probs_test = model.predict_proba(df_test[features])[:, 1]
y_train = df_train['target'].values
y_test = df_test['target'].values

sig_train = y_probs_train[y_train == 1]
bkg_train = y_probs_train[y_train == 0]
sig_test = y_probs_test[y_test == 1]
bkg_test = y_probs_test[y_test == 0]

fig, ax = plt.subplots(figsize=(7, 6))
bins = np.linspace(0, 1, 40)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

h_bkg_train, _ = np.histogram(bkg_train, bins=bins, density=True)
hep.histplot(h_bkg_train, bins=bins, ax=ax, color='red', alpha=0.4, histtype='fill', label='Bkg Train')
h_bkg_test_counts, _ = np.histogram(bkg_test, bins=bins) # Contagens puras para o erro
h_bkg_test_dens, _   = np.histogram(bkg_test, bins=bins, density=True) # Densidade para o plot


scale_bkg = np.divide(h_bkg_test_dens, h_bkg_test_counts, out=np.zeros_like(h_bkg_test_dens), 
                      where=h_bkg_test_counts!=0)
error_bkg = np.sqrt(h_bkg_test_counts) * scale_bkg

ax.errorbar(bin_centers, h_bkg_test_dens, yerr=error_bkg, fmt='o', color='red', 
            markersize=4, label='Bkg Test')



h_sig_train, _ = np.histogram(sig_train, bins=bins, density=True)
hep.histplot(h_sig_train, bins=bins, ax=ax, color='blue', linewidth=2, histtype='step', label='Sig Train')
h_sig_test_counts, _ = np.histogram(sig_test, bins=bins)
h_sig_test_dens, _   = np.histogram(sig_test, bins=bins, density=True)

scale_sig = np.divide(h_sig_test_dens, h_sig_test_counts, out=np.zeros_like(h_sig_test_dens), where=h_sig_test_counts!=0)
error_sig = np.sqrt(h_sig_test_counts) * scale_sig

ax.errorbar(bin_centers, h_sig_test_dens, yerr=error_sig, fmt='o', color='blue', 
            markersize=4, label='Sig Test')

ax.set_yscale('log')
max_y = max(h_sig_train.max(), h_bkg_train.max())
ax.set_ylim(bottom=1e-3, top=max_y * 50) 
ax.set_xlim(0, 1)
ax.set_xlabel("BDT Output Score", fontsize=14)
ax.set_ylabel("a.u.", fontsize=16)
ax.legend(loc='upper center', fontsize=12, ncol=2)

hep.cms.label(ax=ax, label="Work in Progress", data=True,
              lumi=5.01, year="2022C", com=13.6, fontsize=15)

plt.tight_layout()
plt.savefig("overtraining_check.png", dpi=300)
print("Plot salvo como overtraining_check.png")
plt.show()